# 보충 실험 1 — 문헌에서 효과가 있었던 방법을 하나씩 시험

Lab 1~6으로 만든 모델(데이터 정리 + RF 잎 176명, 층화 10-fold AUC 0.856)을 기준으로,
문헌에서 Give Me Some Credit에 효과가 있었다고 보고한 방법을 **한 번에 하나씩** 더해 본다.

| 방법 | 근거 |
|---|---|
| 새 변수 (월 부채액, 1인당 소득, 월 잔액, 사용 비율 1 초과 표시) | Ichim & Issa (ICAART 2025): 새 변수 7개가 모두 중요도 상위 10위 |
| 소득 대체값(6638/6746)을 빈칸으로 되돌림 | 같은 논문: 결측 처리 방식이 성능에 영향 |
| 그래디언트 부스팅 (HistGradientBoosting) | Gidlow 2022 XGBoost 0.867, Kaggle 상위권 |
| Balanced RF (트리마다 연체자·정상을 같은 수로 뽑아 학습) | Ichim & Issa 최고 모델 0.875 |
| 쌓기 (RF + 부스팅 + LR → LR) | Kaggle 2011 1위: 여러 모델 섞기 0.8696 |

평가는 지금까지와 같은 **층화 10-fold (seed 7)**. 같은 시험 조각에서 기준 모델과의 **조각별 차이**를 계산해,
오른 폭이 흔들림보다 큰지 확인한다. (원본 데이터가 아니라 과제용 8개 변수 데이터라 문헌 수치와 직접 비교하지는 않는다.)

In [1]:
import os, time
import numpy
from pandas import read_csv, DataFrame
from joblib import Parallel, delayed
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier, StackingClassifier
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.base import clone

SMOKE = os.environ.get('SMOKE') == '1'
seed = 7
names = ['target', 'revol', 'age', 'late30', 'debt', 'income', 'depend', 'defaulted', 'loans']
features = names[1:]
dataframe = read_csv('../data/givemesomecredit.csv', names=names, header=0)
X = dataframe.values[:, 1:9].astype(float)
Y = dataframe.values[:, 0].astype(int)
if SMOKE:
    X, _, Y, _ = train_test_split(X, Y, train_size=8000, random_state=seed, stratify=Y)
iR, iA, iL, iD, iI, iP, iF, iO = range(8)   # revol, age, late30, debt, income, depend, defaulted, loans
print(X.shape, '· SMOKE' if SMOKE else '')

(150000, 8) 


## 변수 만들기 (모두 행마다 따로 계산 → 학습 데이터 정보를 쓰지 않아 누수 없음)

In [2]:
def clean(A):
    """Lab 5 정리: late30 96/98 → 0 + 표시, 소득 대체값 표시"""
    A = A.copy()
    code = numpy.isin(A[:, iL], [96, 98]).astype(float)
    fill = numpy.isin(A[:, iI], [6638, 6746]).astype(float)
    A[code == 1, iL] = 0
    return numpy.column_stack([A, code, fill])

def engineer(A, income_nan=False):
    """문헌의 새 변수. 소득 대체값 행은 부채 비율 자리에 이미 '금액'이 들어 있다(Lab 1 발견)."""
    B = clean(A)
    fill = B[:, 9] == 1
    income = B[:, iI].copy()
    monthly_debt = numpy.where(fill, B[:, iD], B[:, iD] * income)         # 월 부채액
    per_person = income / (B[:, iP] + 1)                                   # 1인당 소득
    balance = income - monthly_debt                                        # 월 잔액
    revol_over = (B[:, iR] > 1).astype(float)                              # 사용 비율 1 초과
    out = numpy.column_stack([B, monthly_debt, per_person, balance, revol_over])
    if income_nan:                                                         # 대체값을 빈칸으로 (부스팅이 스스로 처리)
        for j in [iI, 11, 12]:                                             # 소득, 1인당 소득, 월 잔액
            out[fill, j] = numpy.nan
    return out

ENG = FunctionTransformer(engineer)
ENG_NAN = FunctionTransformer(engineer, kw_args={'income_nan': True})
CLEAN = FunctionTransformer(clean)
print('정리 후 열', clean(X[:3]).shape[1], '· 새 변수 후 열', engineer(X[:3]).shape[1])

정리 후 열 10 · 새 변수 후 열 14


## Balanced RF 직접 구현 (imbalanced-learn의 BalancedRandomForest와 같은 방식)

In [3]:
def _fit_tree(Xa, y, pos, neg, n, max_features, leaf, rs):
    rng = numpy.random.RandomState(rs)
    idx = numpy.concatenate([rng.choice(pos, n, replace=True), rng.choice(neg, n, replace=True)])
    return DecisionTreeClassifier(max_features=max_features, min_samples_leaf=leaf, random_state=rs).fit(Xa[idx], y[idx])

class BalancedRF(ClassifierMixin, BaseEstimator):
    """트리마다 연체자 n명 + 정상 n명(n = 연체자 수)을 중복 허용으로 뽑아 학습하고 확률을 평균한다."""
    def __init__(self, n_estimators=300, max_features='sqrt', min_samples_leaf=1, random_state=7, n_jobs=-1):
        self.n_estimators, self.max_features, self.min_samples_leaf = n_estimators, max_features, min_samples_leaf
        self.random_state, self.n_jobs = random_state, n_jobs
    def fit(self, X, y):
        X = numpy.asarray(X, dtype=float); y = numpy.asarray(y)
        self.classes_ = numpy.unique(y)
        pos, neg = numpy.where(y == 1)[0], numpy.where(y == 0)[0]
        self.trees_ = Parallel(n_jobs=self.n_jobs)(
            delayed(_fit_tree)(X, y, pos, neg, len(pos), self.max_features, self.min_samples_leaf, self.random_state + k)
            for k in range(self.n_estimators))
        return self
    def predict_proba(self, X):
        X = numpy.asarray(X, dtype=float)
        p = numpy.mean([t.predict_proba(X)[:, 1] for t in self.trees_], axis=0)
        return numpy.column_stack([1 - p, p])
    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)

## 시험할 모델 목록

In [4]:
rf = lambda: RandomForestClassifier(n_estimators=100, max_features=3, min_samples_leaf=176, random_state=seed, n_jobs=-1)
hgb = lambda: HistGradientBoostingClassifier(learning_rate=0.03, max_iter=500, min_samples_leaf=100, l2_regularization=1.0, random_state=seed)
lr = lambda: Pipeline([('std', StandardScaler()), ('lr', LogisticRegression(solver='liblinear', class_weight='balanced'))])
logx = FunctionTransformer(lambda A: numpy.sign(A) * numpy.log1p(numpy.abs(A)))   # 새 변수에 음수(월 잔액)가 있어 부호 유지 로그

models = {
    'A 기준: 정리 + RF (Lab 5)':          Pipeline([('f', CLEAN), ('m', rf())]),
    'B 정리 + 부스팅':                     Pipeline([('f', CLEAN), ('m', hgb())]),
    'C 새 변수 + RF':                      Pipeline([('f', ENG), ('m', rf())]),
    'D 새 변수 + 부스팅':                  Pipeline([('f', ENG), ('m', hgb())]),
    'E 새 변수 + 소득 빈칸 + 부스팅':       Pipeline([('f', ENG_NAN), ('m', hgb())]),
    'F 새 변수 + Balanced RF (잎 1)':      Pipeline([('f', ENG), ('m', BalancedRF(min_samples_leaf=1))]),
    'G 새 변수 + Balanced RF (잎 20)':     Pipeline([('f', ENG), ('m', BalancedRF(min_samples_leaf=20))]),
    'H 새 변수 + 쌓기 (RF·부스팅·LR → LR)': Pipeline([('f', ENG), ('m', StackingClassifier(
        [('rf', rf()), ('hgb', hgb()), ('lr', Pipeline([('log', logx), ('lr', lr())]))],
        final_estimator=LogisticRegression(), cv=3, stack_method='predict_proba'))]),
}
if SMOKE:
    for k, m in models.items():
        if 'Balanced' in k: m.set_params(m__n_estimators=20)

## 층화 10-fold로 같은 조각에서 모두 평가 (조각별 점수를 저장해 짝지어 비교)

In [5]:
kfold = StratifiedKFold(n_splits=(3 if SMOKE else 10), shuffle=True, random_state=seed)
folds = list(kfold.split(X, Y))
OUT = '07_literature_fold_scores.csv' if not SMOKE else '07_smoke.csv'
rows = []
for name, model in models.items():
    t0 = time.time()
    for k, (tr, te) in enumerate(folds):
        m = clone(model).fit(X[tr], Y[tr])
        rows.append((name, k, roc_auc_score(Y[te], m.predict_proba(X[te])[:, 1])))
    sec = time.time() - t0
    s = [r[2] for r in rows if r[0] == name]
    print(f'{name}: AUC {numpy.mean(s):.4f} ({numpy.std(s):.4f}) · {sec / len(folds):.1f}초/조각', flush=True)
    rows_df = DataFrame(rows, columns=['model', 'fold', 'auc']); rows_df.to_csv(OUT, index=False)
    times = globals().setdefault('times', {}); times[name] = sec / len(folds)

A 기준: 정리 + RF (Lab 5): AUC 0.8556 (0.0070) · 4.1초/조각


B 정리 + 부스팅: AUC 0.8569 (0.0068) · 2.3초/조각


C 새 변수 + RF: AUC 0.8554 (0.0069) · 5.5초/조각


D 새 변수 + 부스팅: AUC 0.8573 (0.0066) · 2.5초/조각


E 새 변수 + 소득 빈칸 + 부스팅: AUC 0.8572 (0.0065) · 2.6초/조각


F 새 변수 + Balanced RF (잎 1): AUC 0.8482 (0.0056) · 6.8초/조각


G 새 변수 + Balanced RF (잎 20): AUC 0.8561 (0.0064) · 4.7초/조각


H 새 변수 + 쌓기 (RF·부스팅·LR → LR): AUC 0.8525 (0.0069) · 31.6초/조각


## 기준 모델(A)과의 조각별 차이

In [6]:
df = DataFrame(rows, columns=['model', 'fold', 'auc']).pivot(index='fold', columns='model', values='auc')
base = df['A 기준: 정리 + RF (Lab 5)']
summary = DataFrame({
    'AUC': df.mean(), '흔들림': df.std(ddof=0),
    '기준 대비': df.sub(base, axis=0).mean(),
    '차이의 흔들림': df.sub(base, axis=0).std(ddof=0),
    '이긴 조각 수': (df.sub(base, axis=0) > 0).sum(),
    '초/조각': DataFrame.from_dict(times, orient='index')[0],
}).sort_values('AUC', ascending=False)
print(summary.round(4).to_string())

                                 AUC     흔들림   기준 대비  차이의 흔들림  이긴 조각 수     초/조각
D 새 변수 + 부스팅                  0.8573  0.0066  0.0016   0.0011        9   2.5464
E 새 변수 + 소득 빈칸 + 부스팅          0.8572  0.0065  0.0016   0.0010        9   2.5966
B 정리 + 부스팅                    0.8569  0.0068  0.0013   0.0011        9   2.3278
G 새 변수 + Balanced RF (잎 20)   0.8561  0.0064  0.0005   0.0010        8   4.7087
A 기준: 정리 + RF (Lab 5)         0.8556  0.0070  0.0000   0.0000        0   4.1118
C 새 변수 + RF                   0.8554  0.0069 -0.0002   0.0008        4   5.5237
H 새 변수 + 쌓기 (RF·부스팅·LR → LR)  0.8525  0.0069 -0.0031   0.0010        0  31.6139
F 새 변수 + Balanced RF (잎 1)    0.8482  0.0056 -0.0074   0.0016        0   6.8242


읽는 법: '기준 대비'가 '차이의 흔들림'보다 확실히 크고(대략 2배 이상), 10조각 중 대부분에서 이겼다면 의미 있게 오른 것이다.
그렇지 않으면 우연과 구별하기 어렵다.

## 새 변수의 중요도 (새 변수 + RF를 전체 데이터로 한 번 학습)

In [7]:
fnames = features + ['late30_코드', '소득_대체값', '월부채액', '1인당소득', '월잔액', '사용비율>1']
m = Pipeline([('f', ENG), ('m', rf())]).fit(X, Y)
imp = DataFrame({'중요도': m.named_steps['m'].feature_importances_}, index=fnames).sort_values('중요도', ascending=False)
print(imp.round(3).to_string())

             중요도
defaulted  0.333
revol      0.287
late30     0.159
사용비율>1     0.074
age        0.030
월잔액        0.023
월부채액       0.022
income     0.019
debt       0.018
1인당소득      0.017
loans      0.013
depend     0.004
late30_코드  0.002
소득_대체값     0.001


## 정리
- **부스팅만 일관되게 조금 높았다.** 새 변수 + 부스팅 0.8573으로 기준(0.8556)보다 +0.0016, 10조각 중 9조각에서 이겼다(차이의 흔들림 0.0011). 우연은 아니지만 폭은 매우 작다. 학습도 RF보다 빠르다(조각당 2.5초 vs 4.1초).
- **새 변수 효과는 거의 없었다.** RF에서는 -0.0002, 부스팅에서는 +0.0004. '사용 비율 1 초과' 표시만 중요도 4위였다. 원본에서 효과가 컸던 변수(최대 연체, 총 연체 횟수)는 이 데이터가 연체 열을 합쳐 두어 만들 수 없었다.
- **소득 대체값을 빈칸으로 되돌려도 차이 없음** (0.8572).
- **Balanced RF는 잎 크기에 따라 갈렸다.** 잎 20명 0.8561(+0.0005), 잎 1명 0.8482(-0.0074). 문헌의 0.875는 원본 데이터·이상값 제거 후의 값이라 직접 비교할 수 없다.
- **쌓기는 오히려 낮았다** (0.8525, 10조각 모두 짐). 안쪽 3-fold로 나눠 학습한 기본 모델들이 약해진 것으로 보인다.

→ 최적 모델은 **새 변수 + 부스팅**을 바탕으로 설정을 더 넓게 찾는다(09 노트북).